In [31]:
pip install requests

In [39]:
import requests

SERVICE_KEY = "a71451c7c7109143b19101cf7ff0fcb04f7278bf1424454a720d9a9d1db98ecb"
CITY_CODE = "32010"  # 춘천시

# 1. 좌표 기반 근접 정류소 조회 API (반경 500m 자동 검색)
NEARBY_STATION_API = "http://apis.data.go.kr/1613000/BusSttnInfoInqireService/getCrdntPrxmtSttnList"
# 2. 버스 도착 정보 조회 API
ARRIVAL_API = "http://apis.data.go.kr/1613000/ArvlInfoInqireService/getSttnAcctoArvlPrearngeInfoList"

# 한림대학교 중심 GPS 좌표 (WGS84)
HALLYM_LAT = "37.88635"  # 위도 (gpsLati)
HALLYM_LON = "127.73783"  # 경도 (gpsLong)


def get_nearby_stations(lat, lon):
    """GPS 좌표 기준 500m 이내 정류소 목록 가져오기"""
    url = f"{NEARBY_STATION_API}?serviceKey={SERVICE_KEY}&gpsLati={lat}&gpsLong={lon}&numOfRows=50&pageNo=1&_type=json"

    res = requests.get(url, timeout=10)
    data = res.json()

    body = data.get("response", {}).get("body", {})
    if not body:
        return []

    items_container = body.get("items")
    if not items_container or not isinstance(items_container, dict):
        return []

    items = items_container.get("item", [])
    if isinstance(items, dict):
        items = [items]

    return items


def get_arrival_info(node_id, node_name):
    """각 정류소별 실시간 버스 도착 정보 조회"""
    url = f"{ARRIVAL_API}?serviceKey={SERVICE_KEY}&cityCode={CITY_CODE}&nodeId={node_id}&numOfRows=20&pageNo=1&_type=json"

    try:
        res = requests.get(url, timeout=10)
        data = res.json()

        body = data.get("response", {}).get("body", {})
        items = body.get("items")

        print(f"\n📍 [{node_name}] (정류소 ID: {node_id})")

        if not items or not isinstance(items, dict):
            print("   -> 현재 운행/도착 예정인 버스가 없습니다.")
            return

        item_list = items.get("item", [])
        if isinstance(item_list, dict):
            item_list = [item_list]

        for item in item_list:
            route_no = item.get("routeno")
            arr_time = item.get("arrtime", 0)
            prev_cnt = item.get("arrprevstationcnt", 0)
            vehicle_tp = item.get("vehicletp", "일반버스")

            min_left = arr_time // 60
            sec_left = arr_time % 60

            print(
                f"  - {route_no}번 ({vehicle_tp}): 약 {min_left}분 {sec_left}초 후 ({prev_cnt}개 전 정류장)"
            )

    except Exception as e:
        print(f"  -> 도착 정보 조회 오류: {e}")


if __name__ == "__main__":
    print("=== 한림대학교 반경 500m 이내 버스 정류소 검색 및 도착 정보 ===")

    stations = get_nearby_stations(HALLYM_LAT, HALLYM_LON)

    if not stations:
        print("반경 500m 이내에 검색된 정류소가 없습니다.")
    else:
        print(
            f"한림대 반경 500m 내에서 총 {len(stations)}개의 정류소를 찾았습니다.\n"
        )
        print("-" * 60)

        for sttn in stations:
            node_id = sttn.get("nodeid")
            node_name = sttn.get("nodenm")
            get_arrival_info(node_id, node_name)

=== 한림대학교 반경 500m 이내 버스 정류소 검색 및 도착 정보 ===
한림대 반경 500m 내에서 총 5개의 정류소를 찾았습니다.

------------------------------------------------------------

📍 [한림대학교] (정류소 ID: CCB250002127)
  - 2번 (저상버스): 약 3분 0초 후 (5개 전 정류장)
  - 300번 (저상버스): 약 12분 0초 후 (16개 전 정류장)

📍 [한림대학교] (정류소 ID: CCB250001275)
  - 2번 (저상버스): 약 13분 0초 후 (19개 전 정류장)
  - 10-1번 (저상버스): 약 15분 0초 후 (21개 전 정류장)
  - 7-2(칠전동경유)번 (저상버스): 약 27분 0초 후 (30개 전 정류장)

📍 [동보빌리지] (정류소 ID: CCB250001547)
  - 2번 (저상버스): 약 3분 0초 후 (4개 전 정류장)

📍 [동보빌리지] (정류소 ID: CCB250001548)
  - 2번 (저상버스): 약 14분 0초 후 (20개 전 정류장)
  - 10-1번 (저상버스): 약 16분 0초 후 (22개 전 정류장)
  - 7-2(칠전동경유)번 (저상버스): 약 28분 0초 후 (31개 전 정류장)

📍 [동산A] (정류소 ID: CCB250001540)
  - 2번 (저상버스): 약 0분 0초 후 (1개 전 정류장)
